In [2]:
# =============================================================================
# Model 2 — Random Forest Regressor
# Task    : Predict Crime_Count based on Location and Date
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df = pd.read_csv("./crime_per_capita_df_corrected.csv")

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs: {df['LSOA_Code'].nunique()}")

# =============================================================================
# 2. FEATURE SELECTION
# NOTE: Random Forest does NOT need scaling (tree-based model)
# =============================================================================
FEATURES = [
    # --- Location ---
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",

    # --- Date ---
    "Year",
    "Month",
    "month_sin",
    "month_cos",

    # --- Season flags ---
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",

    # --- Socioeconomic ---
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",

    # --- Weather ---
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",

    # --- Lag & rolling features ---
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
]

TARGET = "Crime_Count"

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

print(f"\nUsing {len(FEATURES)} features → target: '{TARGET}'")

# =============================================================================
# 3. TIME-BASED TRAIN / TEST SPLIT
# =============================================================================
train = df[df["Year"].isin([2020, 2021, 2022, 2023])].copy()
test  = df[df["Year"] == 2024].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train : {len(X_train):,} rows  |  Test : {len(X_test):,} rows")

# =============================================================================
# 4. EVALUATION HELPER
# =============================================================================
def evaluate(model_name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R²    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {"Model": model_name, "MAE": round(mae,4), "RMSE": round(rmse,4),
            "R2": round(r2,4), "MAPE": round(mape,2)}

# =============================================================================
# 5. BASELINE RANDOM FOREST
# Quick run with sensible defaults before tuning
# n_jobs=-1 uses all CPU cores
# =============================================================================
print("\n--- Fitting Baseline Random Forest ---")

rf_baseline = RandomForestRegressor(
    n_estimators=100,       # number of trees
    max_depth=None,         # grow fully — we'll constrain later
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf_baseline.fit(X_train, y_train)
preds_baseline = np.clip(rf_baseline.predict(X_test), 0, None)
res_baseline = evaluate("RF — baseline (100 trees, no depth limit)", y_test, preds_baseline)

# =============================================================================
# 6. TUNED VARIANTS
# We try 3 configurations to understand the depth/trees tradeoff
# This replaces a full GridSearch which would take hours on this dataset
# =============================================================================
configs = [
    {"n_estimators": 200, "max_depth": 10,   "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 20,   "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 4},
]

results = [res_baseline]

for cfg in configs:
    label = (f"RF — {cfg['n_estimators']} trees, "
             f"depth={cfg['max_depth']}, leaf={cfg['min_samples_leaf']}")
    print(f"\n--- Fitting {label} ---")

    rf = RandomForestRegressor(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        min_samples_leaf=cfg["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    preds = np.clip(rf.predict(X_test), 0, None)
    res = evaluate(label, y_test, preds)
    results.append({**res, "model_obj": rf})

# Attach baseline model object for feature importance
results[0]["model_obj"] = rf_baseline

# =============================================================================
# 7. PICK BEST MODEL
# =============================================================================
best = min(results, key=lambda x: x["MAE"])
best_rf = best["model_obj"]
print(f"\nBest configuration: {best['Model']}")
print(f"MAE={best['MAE']}  RMSE={best['RMSE']}  R²={best['R2']}  MAPE={best['MAPE']}%")

# =============================================================================
# 8. FEATURE IMPORTANCE
# RF gives native importance scores based on mean decrease in impurity
# =============================================================================
print("\n--- Feature Importances (Top 15) ---")
fi = pd.DataFrame({
    "Feature"   : FEATURES,
    "Importance": best_rf.feature_importances_
}).sort_values("Importance", ascending=False)

print(fi.head(15).to_string(index=False))

# Warn if lag features dominate (expected, but good to quantify)
lag_cols   = [f for f in FEATURES if "lag" in f or "rolling" in f or "msoa_avg" in f]
lag_imp    = fi[fi["Feature"].isin(lag_cols)]["Importance"].sum()
other_imp  = 1 - lag_imp
print(f"\nLag/rolling features combined importance : {lag_imp*100:.1f}%")
print(f"All other features combined importance   : {other_imp*100:.1f}%")

# =============================================================================
# 9. WITHOUT LAG FEATURES — test the honest performance
# This shows how much the model can predict from location + date alone
# =============================================================================
print("\n--- Fitting RF WITHOUT lag features (honest spatial model) ---")

NO_LAG_FEATURES = [f for f in FEATURES if f not in lag_cols]
print(f"Features used (no lags): {len(NO_LAG_FEATURES)}")

rf_no_lag = RandomForestRegressor(
    n_estimators=200, max_depth=20,
    min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf_no_lag.fit(X_train[NO_LAG_FEATURES], y_train)
preds_no_lag = np.clip(rf_no_lag.predict(X_test[NO_LAG_FEATURES]), 0, None)
res_no_lag = evaluate("RF — no lag features (spatial only)", y_test, preds_no_lag)
results.append({**res_no_lag, "model_obj": rf_no_lag})

# =============================================================================
# 10. PREDICTION INTERVALS via individual tree predictions
# RF = ensemble of trees → std of tree predictions ≈ uncertainty estimate
# =============================================================================
print("\n--- Prediction Intervals (uncertainty estimate) ---")
all_tree_preds = np.array([
    tree.predict(X_test) for tree in best_rf.estimators_
])
pred_mean = all_tree_preds.mean(axis=0)
pred_std  = all_tree_preds.std(axis=0)
pred_low  = np.clip(pred_mean - 1.96 * pred_std, 0, None)
pred_high = pred_mean + 1.96 * pred_std

sample_idx = test.head(8).index
sample_df  = test.loc[sample_idx, ["LSOA_Code", "Year", "Month", TARGET]].copy()
sample_df["Predicted"] = np.round(pred_mean[:8], 1)
sample_df["Lower_95"]  = np.round(pred_low[:8],  1)
sample_df["Upper_95"]  = np.round(pred_high[:8], 1)
sample_df["Actual"]    = y_test.values[:8]
print(sample_df.to_string(index=False))

# =============================================================================
# 11. ERROR ANALYSIS — where does the model struggle?
# =============================================================================
print("\n--- Error Analysis ---")
error_df = test[["LSOA_Code", "Year", "Month", TARGET]].copy()
error_df["Predicted"] = np.round(pred_mean, 1)
error_df["Error"]     = np.abs(error_df[TARGET] - error_df["Predicted"])
error_df["Pct_Error"] = (error_df["Error"] / (error_df[TARGET] + 1e-9)) * 100

# LSOAs with highest average error
worst_lsoas = (error_df.groupby("LSOA_Code")["Error"]
               .mean().sort_values(ascending=False).head(10))
print("\nTop 10 LSOAs with highest average prediction error:")
print(worst_lsoas.to_string())

# Error by month
monthly_err = (error_df.groupby("Month")["Error"]
               .mean().reset_index()
               .rename(columns={"Error": "Avg_MAE"}))
print("\nAverage error by month (1=Jan … 12=Dec):")
print(monthly_err.to_string(index=False))

# =============================================================================
# 12. APPEND TO MODEL COMPARISON FILE
# =============================================================================
new_rows = pd.DataFrame([
    {k: v for k, v in r.items() if k != "model_obj"}
    for r in results
])

try:
    existing = pd.read_csv("model_comparison.csv")
    combined = pd.concat([existing, new_rows], ignore_index=True)
except FileNotFoundError:
    combined = new_rows

combined.to_csv("model_comparison.csv", index=False)
print("\n\nUpdated model_comparison.csv:")
print(combined[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))


# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI — Prediction Accuracy Index
# RRI — Recapture Rate Index
# PEI — Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10):

    df_eval = pd.DataFrame({
        "actual": y_true,
        "predicted": y_pred
    }).reset_index(drop=True)

    # Select hotspot cells (top X% predicted crime)
    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    # Crime counts
    crimes_in_hotspots = df_eval.loc[df_eval["hotspot"], "actual"].sum()
    total_crimes = df_eval["actual"].sum()

    # Area proportions
    hotspot_area = df_eval["hotspot"].sum()
    total_area = len(df_eval)

    # Metrics
    RRI = crimes_in_hotspots / total_crimes
    area_ratio = hotspot_area / total_area
    PAI = RRI / area_ratio
    PEI = RRI / area_ratio   # simplified PEI approximation

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {RRI:.4f}")
    print(f"PAI               : {PAI:.4f}")
    print(f"PEI               : {PEI:.4f}")

    return {
        "Hotspot_%": hotspot_percent,
        "RRI": round(RRI,4),
        "PAI": round(PAI,4),
        "PEI": round(PEI,4)
    }

# =============================================================================
# 13. CRIME HOTSPOT EVALUATION
# =============================================================================
print("\n--- Crime Forecasting Indices ---")

hotspot_results = []

for pct in [5, 10, 20]:
    res = crime_hotspot_metrics(y_test.values, pred_mean, pct)
    hotspot_results.append(res)

hotspot_df = pd.DataFrame(hotspot_results)

print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))

Dataset shape: (304336, 39)
Date range: 2020 to 2024
Unique LSOAs: 12415

Using 30 features → target: 'Crime_Count'
Train : 241,710 rows  |  Test : 62,626 rows

--- Fitting Baseline Random Forest ---

  RF — baseline (100 trees, no depth limit)
  MAE   : 4.4455
  RMSE  : 8.6718
  R²    : 0.9423
  MAPE  : 41.04%

--- Fitting RF — 200 trees, depth=10, leaf=2 ---

  RF — 200 trees, depth=10, leaf=2
  MAE   : 4.3771
  RMSE  : 8.1814
  R²    : 0.9486
  MAPE  : 40.29%

--- Fitting RF — 200 trees, depth=20, leaf=2 ---

  RF — 200 trees, depth=20, leaf=2
  MAE   : 4.3825
  RMSE  : 8.1720
  R²    : 0.9487
  MAPE  : 40.04%

--- Fitting RF — 300 trees, depth=None, leaf=4 ---

  RF — 300 trees, depth=None, leaf=4
  MAE   : 4.3975
  RMSE  : 8.4439
  R²    : 0.9453
  MAPE  : 40.30%

Best configuration: RF — 200 trees, depth=10, leaf=2
MAE=4.3771  RMSE=8.1814  R²=0.9486  MAPE=40.29%

--- Feature Importances (Top 15) ---
                Feature  Importance
           crime_lag_1m    0.546811
  crime_r